# Notebook 10: Data Loading and Aggregation

Loads raw mastery attempt data from mastery_data.xlsx and builds early-window (Weeks 1-3) student features. The resulting feature table is exported for use in Notebook 11.


In [1]:
import pandas as pd 
import numpy as np

In [2]:
file_path = "mastery_data.xlsx"

In [3]:
attempts_df = pd.read_excel(file_path, sheet_name="All Attempts")

In [4]:
attempts_df['attempt_date'] = pd.to_datetime(attempts_df['attempt_date'])

In [5]:
attempts_df.head(), attempts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14338 entries, 0 to 14337
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   new ID          14338 non-null  object        
 1   year            14338 non-null  int64         
 2   attempt_date    14338 non-null  datetime64[ns]
 3   progression     14338 non-null  int64         
 4   pass/fail       14338 non-null  int64         
 5   attempt_number  14338 non-null  int64         
 6   week_number     14338 non-null  int64         
dtypes: datetime64[ns](1), int64(5), object(1)
memory usage: 784.2+ KB


(     new ID  year        attempt_date  progression  pass/fail  attempt_number  \
 0  EWWZ8744  2014 2014-03-03 10:11:00            1          1               1   
 1  FONP9675  2014 2014-03-03 10:12:00            1          1               1   
 2  RNXB3999  2014 2014-03-03 10:12:00            1          1               1   
 3  QSJZ8609  2014 2014-03-03 10:15:00            1          1               1   
 4  VRMY8240  2014 2014-03-03 10:16:00            1          1               1   
 
    week_number  
 0            1  
 1            1  
 2            1  
 3            1  
 4            1  ,
 None)

In [6]:
attempts_df= attempts_df.rename(columns={ 'new ID': 'id', 
                                          'year':'year', 
                                          'attempt_date': 'attempt_date', 
                                          'progression':'level', 
                                          'pass/fail': 'pass_fail', 
                                          'attempt_number': 'attempt_num',
                                          'week_number': 'week' }) 

attempts_df.head(), attempts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14338 entries, 0 to 14337
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id            14338 non-null  object        
 1   year          14338 non-null  int64         
 2   attempt_date  14338 non-null  datetime64[ns]
 3   level         14338 non-null  int64         
 4   pass_fail     14338 non-null  int64         
 5   attempt_num   14338 non-null  int64         
 6   week          14338 non-null  int64         
dtypes: datetime64[ns](1), int64(5), object(1)
memory usage: 784.2+ KB


(         id  year        attempt_date  level  pass_fail  attempt_num  week
 0  EWWZ8744  2014 2014-03-03 10:11:00      1          1            1     1
 1  FONP9675  2014 2014-03-03 10:12:00      1          1            1     1
 2  RNXB3999  2014 2014-03-03 10:12:00      1          1            1     1
 3  QSJZ8609  2014 2014-03-03 10:15:00      1          1            1     1
 4  VRMY8240  2014 2014-03-03 10:16:00      1          1            1     1,
 None)

# 14338 total attempts in full dataset

In [7]:
# Remove spaces in IDs
attempts_df['id'] = attempts_df['id'].str.strip()

# Checks for reattempts on a level after it was completed

In [8]:
attempts_df['attempt_date'].dtype

dtype('<M8[ns]')

In [9]:
first_pass = (
    attempts_df[attempts_df['pass_fail'] == 1]
    .groupby(['id','year','level'])['attempt_date']
    .min()
    .reset_index()
    .rename(columns={'attempt_date':'first_pass_date'})
)

attempts_check = attempts_df.merge(
    first_pass,
    on=['id','year','level'],
    how='left'
)

reattempts_after_pass = attempts_check[
    attempts_check['attempt_date'] > attempts_check['first_pass_date']
]

len(reattempts_after_pass), reattempts_after_pass

(1,
              id  year        attempt_date  level  pass_fail  attempt_num  \
 14334  VAGR7856  2018 2018-06-05 09:20:37     10          0            1   
 
        week     first_pass_date  
 14334    13 2018-05-29 11:23:54  )

In [10]:
reattempts_after_pass[['id','year','level','attempt_date','first_pass_date','pass_fail','attempt_num']]

,id,year,level,attempt_date,first_pass_date,pass_fail,attempt_num
14334,VAGR7856,2018,10,2018-06-05 09:20:37,2018-05-29 11:23:54,0,1


The record above shows that there was a reattempt by one student after passing level 10 but this attempt was failed and is recorded as attempt 1, which is not the case, so this is a system error. I will leave it since this attempt is in the later weeks.

92 reattempts after mastery were initially detected. Further inspection showed these cases were caused by students appearing in multiple cohort years. So a data integrity check was conducted to verify that students did not attempt mastery levels after successfully passing them within separate cohorts. After accounting for cohort year, only one instance of a post-mastery attempt was identified among 14,338 attempts. Inspection showed that the attempt number was reset, suggesting a system logging issue rather than a genuine progression violation. This occurred in week 13 of the semester and does not affect early semester predictions. Given its negligible frequency, the record was retained. New problem: How do we treat the repeated IDs since we created the master df from unique IDs? Group ids with cohort year

In [11]:
# Early attempts dataset (attempts made in Weeks 1-3)
early_attempts = attempts_df[attempts_df['week'] <= 3].copy()

# Dropping attempt_date column (not needed since we have week column)
early_attempts = early_attempts[['id','year','level','pass_fail','attempt_num','week']]

early_attempts.head(), early_attempts.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3642 entries, 0 to 12259
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           3642 non-null   object
 1   year         3642 non-null   int64 
 2   level        3642 non-null   int64 
 3   pass_fail    3642 non-null   int64 
 4   attempt_num  3642 non-null   int64 
 5   week         3642 non-null   int64 
dtypes: int64(5), object(1)
memory usage: 199.2+ KB


(         id  year  level  pass_fail  attempt_num  week
 0  EWWZ8744  2014      1          1            1     1
 1  FONP9675  2014      1          1            1     1
 2  RNXB3999  2014      1          1            1     1
 3  QSJZ8609  2014      1          1            1     1
 4  VRMY8240  2014      1          1            1     1,
 None)

# The output above shows 3,642 early attempts <= week 3 (of 14,338 attempts total); approx. 25% of all attempts.

In [12]:
#Total number of students that made attempts in the full dataset
num_students= attempts_df['id'].nunique()
num_students

1135

There are 1135 records of individual students in the full dataset, but what about those students who repeat the course? 

In [13]:
# To check for number of students who took the course more than once
repeat_counts = attempts_df[['id','year']].drop_duplicates()['id'].value_counts()

repeat_counts[repeat_counts > 1], (repeat_counts > 1).sum()

(id
 PWPO9424    3
 BGMN5662    2
 TXRA4860    2
 UQEB3636    2
 PEIQ9634    2
 GQTE6968    2
 DBVM4419    2
 WDME3259    2
 FHAR8432    2
 GYZL8284    2
 MHDM5723    2
 JVAL5945    2
 ANND1245    2
 KVAW4798    2
 QWNT7178    2
 ETUL8523    2
 FKJJ0623    2
 RCQU7334    2
 Name: count, dtype: int64,
 np.int64(18))

This tell us that 17 students retook the course once while 1 student retook the course twice, resulting in 19 additional enrolments

In [14]:
# Total number of student-cohort records (unique id-year pairs) in the mastery attempts dataset

total_students = attempts_df[['id','year']].drop_duplicates().shape[0]
total_students

1154

The mastery-attempt dataset contained 1,135 unique student IDs. However, when accounting for cohort year, 1,154 unique student-cohort instances were identified. This confirms that some students appeared in multiple cohort years, representing repeated enrolments in the course. Therefore, the unit of analysis was defined as the student-cohort instance (id-year), ensuring that each course attempt was treated as a separate observation.

In [15]:
# Total students with attempts in the first 3 weeks
early_students = early_attempts[['id','year']].drop_duplicates().shape[0]
early_students

1133

# The value above shows that there are 1133 students who made mastery test attempts in Weeks 1-3

In [16]:
# Students with NO attempts in the first 3 weeks
inactive_students_3w = total_students - early_students

inactive_students_3w

21

21 student-cohort instances made mastery attempts in the course but did not attempt any mastery tests during Weeks 1-3

In [17]:
# Getting the IDs of students with NO attempts in Weeks 1-3 (student-cohort level)

all_students = set(
    tuple(x) for x in attempts_df[['id','year']].drop_duplicates().values
)

early_active_students = set(
    tuple(x) for x in early_attempts[['id','year']].drop_duplicates().values
)

inactive_students = all_students - early_active_students

len(inactive_students), list(inactive_students)[:10]

(21,
 [('SLLP1898', 2016),
  ('OBWM7893', 2016),
  ('LRLB2557', 2016),
  ('RVMF4552', 2015),
  ('BYRN8962', 2014),
  ('GYIY5442', 2017),
  ('FCEI8433', 2017),
  ('KIAM7733', 2016),
  ('SVNP1242', 2016),
  ('XEQB3620', 2017)])

The IDs of the 21 student-cohort instances that were inactive in Weeks 1-3 are listed above. In summary, of the 1,154 student-cohort instances in the mastery attempts dataset, 1,133 engaged with mastery tests during the first three weeks, while 21 were inactive during this period.

In [18]:
# Master students dataframe to ensure each student-cohort instance occurs only once

# Create one row per unique student-cohort instance
master_students = (
    attempts_df[['id', 'year']]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(master_students.shape)
print(master_students[['id', 'year']].drop_duplicates().shape[0])

(1154, 2)
1154


# The master_students dataframe serves as the backbone for merging all early-student features. It ensures that no student is lost in the aggregation process, including those who were inactive during Weeks 1-3.

In [19]:
# Unique attempts in full dataset
attempts_df['level'].unique() 

array([ 1,  2,  3,  4,  5,  6,  7,  8, 11,  9, 12, 10])

In [20]:
# Unique attempts in Weeks 1-3
early_attempts['level'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8, 11])

In [21]:
attempts_df['level'].value_counts().sort_index()

level
1     1337
2     1929
3     1885
4     2576
5     2500
6     1955
7     1433
8      396
9      141
10      90
11      66
12      30
Name: count, dtype: int64

In [22]:
# Filtering passed attempts for weeks 1-3 
early_passes = early_attempts[early_attempts['pass_fail'] == 1] 

# Counting unique students per level 
students_per_level = (
    early_passes[['id', 'year', 'level']]
    .drop_duplicates()
    .groupby('level')
    .size()
    .sort_index()
)
students_per_level # weeks 1-3

level
1    1118
2     871
3     445
4      61
5      18
6       5
7       2
8       2
dtype: int64

Although early completions of Levels 4-8 were rare, they were retained to preserve potentially informative signals of rapid progression.

In [23]:
# Number of attempts per level in Weeks 1-3 including all levels attempted during this period.
# A value of 0 means that the level was not attempted in Weeks 1-3.

attempts_per_level = ( early_attempts.groupby(['id','year','level'])['attempt_num'] .max() .unstack(fill_value=0) ) 

# Rename columns 
attempts_per_level.columns = [f'num_atmp_l{int(col)}' for col in attempts_per_level.columns] 

attempts_per_level.head()

,,num_atmp_l1,num_atmp_l2,num_atmp_l3,num_atmp_l4,num_atmp_l5,num_atmp_l6,num_atmp_l7,num_atmp_l8,num_atmp_l11
id,year,,,,,,,,,
ABEQ5474,2016,1,2,0,0,0,0,0,0,0
ACFZ9682,2016,1,1,1,0,0,0,0,0,0
ADIO2423,2017,1,1,1,0,0,0,0,0,0
AEWT6889,2014,1,1,1,0,0,0,0,0,0
AFPG1281,2014,1,2,1,0,0,0,0,0,0


In [24]:
# Week of completion for all levels completed in Weeks 1-3, 0 means that the level was not completed in Weeks 1-3
completion_full = (
    attempts_df[attempts_df['pass_fail'] == 1]
    .groupby(['id','year','level'])['week']
    .min()
    .unstack()
)

# Keep only completions that occurred in Weeks 1-3
completion_3w = completion_full.where(completion_full <= 3)

# Rename columns
completion_3w.columns = [f'wk_comp_l{int(col)}' for col in completion_3w.columns]

# Keep only levels where at least one student completed the level in Weeks 1-3
completion_3w = completion_3w.loc[:, completion_3w.notna().any()]

# Replace NaN with 0
completion_3w = completion_3w.fillna(0)

completion_3w.head()

,,wk_comp_l1,wk_comp_l2,wk_comp_l3,wk_comp_l4,wk_comp_l5,wk_comp_l6,wk_comp_l7,wk_comp_l8
id,year,,,,,,,,
ABEQ5474,2016,1.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0
ACFZ9682,2016,1.0,2.0,3.0,0.0,0.0,0.0,0.0,0.0
ADIO2423,2017,2.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0
AEWT6889,2014,1.0,2.0,3.0,0.0,0.0,0.0,0.0,0.0
AFPG1281,2014,1.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
# Gaps / periods of inactivity between consecutive level completions in Weeks 1-3

# Get all week-of-completion columns in order
wk_comp_cols = sorted(
    [col for col in completion_3w.columns if col.startswith('wk_comp_l')],
    key=lambda x: int(x.replace('wk_comp_l', ''))
)

# Create gap features between consecutive completed levels
gap_features = pd.DataFrame(index=completion_3w.index)

for i in range(len(wk_comp_cols) - 1):
    current_col = wk_comp_cols[i]
    next_col = wk_comp_cols[i + 1]

    current_level = current_col.replace('wk_comp_l', '')
    next_level = next_col.replace('wk_comp_l', '')

    gap_name = f'gap{current_level}to{next_level}_3w'

    gap = completion_3w[next_col] - completion_3w[current_col]
    gap = gap.clip(lower=0).fillna(0)

    gap_features[gap_name] = gap

# Total inactivity in Weeks 1-3
total_inactivity_3w = gap_features.sum(axis=1).rename('total_inactivity_3w')

gap_features.head(), total_inactivity_3w.head()

(               gap1to2_3w  gap2to3_3w  gap3to4_3w  gap4to5_3w  gap5to6_3w  \
 id       year                                                               
 ABEQ5474 2016         2.0         0.0         0.0         0.0         0.0   
 ACFZ9682 2016         1.0         1.0         0.0         0.0         0.0   
 ADIO2423 2017         1.0         0.0         0.0         0.0         0.0   
 AEWT6889 2014         1.0         1.0         0.0         0.0         0.0   
 AFPG1281 2014         1.0         0.0         0.0         0.0         0.0   
 
                gap6to7_3w  gap7to8_3w  
 id       year                          
 ABEQ5474 2016         0.0         0.0  
 ACFZ9682 2016         0.0         0.0  
 ADIO2423 2017         0.0         0.0  
 AEWT6889 2014         0.0         0.0  
 AFPG1281 2014         0.0         0.0  ,
 id        year
 ABEQ5474  2016    2.0
 ACFZ9682  2016    2.0
 ADIO2423  2017    1.0
 AEWT6889  2014    2.0
 AFPG1281  2014    1.0
 Name: total_inactivity_3w, dtype

Gaps between successive levels were calculated using first completion weeks derived from the full dataset to preserve sequential ordering. Only gaps of completions occurring within Weeks 1-3 were considered. When a subsequent level was not completed within Weeks 1-3, the resulting negative difference was clipped to zero, indicating no measurable early progression delay.

Initial Issue: When calculating gaps between successive mastery levels (e.g., Level 2 - Level 1), some gaps were negative. This happened because the completion weeks were initially computed using only attempts within Weeks 1-3. For students who completed Level 1 before Week 1-3 but Level 2 within Weeks 1-3, the filtered dataset could incorrectly suggest that Level 2 was completed before Level 1, producing negative values. These were not true violations of the mastery sequence, but indications of early filtering.

Correction: First completion week for each level was calculated using the full dataset, preserving the natural sequential order.
The early prediction window (Weeks 1-3) was applied after computing true first completion weeks.
Gaps were then calculated only for completions occurring within Weeks 1-3.
Negative gaps were clipped to zero, a

In [26]:
# Total number of attempts in Weeks 1-3

total_attempts_3w = (
    early_attempts
    .groupby(['id','year'])
    .size()
    .rename('total_attempts_3w')
)
total_attempts_3w.head()

id        year
ABEQ5474  2016    3
ACFZ9682  2016    3
ADIO2423  2017    3
AEWT6889  2014    3
AFPG1281  2014    4
Name: total_attempts_3w, dtype: int64

In [27]:
# Total number of levels completed in Weeks 1-3
levels_completed_3w = (
    early_attempts[early_attempts['pass_fail'] == 1]
    .groupby(['id','year'])['level']
    .nunique()
    .rename('levels_completed_3w')
)
levels_completed_3w.head()

id        year
ABEQ5474  2016    2
ACFZ9682  2016    3
ADIO2423  2017    3
AEWT6889  2014    3
AFPG1281  2014    2
Name: levels_completed_3w, dtype: int64

In [28]:
# Efficiency/ Difficulty Ratio for Weeks 1-3: total levels completed in weeks 1-3 divided by total attempts in weeks 1-3

efficiency_3w = (levels_completed_3w / total_attempts_3w).replace([np.inf, np.nan], 0).rename('efficiency_3w')
efficiency_3w.head()

id        year
ABEQ5474  2016    0.666667
ACFZ9682  2016    1.000000
ADIO2423  2017    1.000000
AEWT6889  2014    1.000000
AFPG1281  2014    0.500000
Name: efficiency_3w, dtype: float64

Efficiency was set to zero for students with no early attempts.
efficiency_3w measures how effectively a student converts attempts into successful level completions during the first three weeks.

For example: 
3 levels completed in 3 attempts, high mastery, efficiency = 1.00
3 levels completed in 5 attempts, moderate mastery,efficiency = 0.60
1 level completed in 5 attempts, low mastery, 0.20

So higher values indicate that a student:
requires fewer attempts to pass levels
progresses smoothly through the mastery system
likely understands the material earlier.

The efficiency ratio can produce misleading interpretations if used alone.

Example	 		
Student A has 3 attempts 3 with 3 levels completed so efficiency=1
Student B has 10 attempts with 5 levels completed so efficiency=0.50

The metric suggests Student A is better, but that may not be true.

Student B actually:progressed further, attempted more content, and may be more engaged.
The ratio penalises students who progress further but struggle slightly, confusing progress depth  vs  attempt difficulty. 

Students who made no attempts produce: 0/0 which is undefined, so it is replace with 0 which means inactive student receive 0 which does not represent real inefficiency value. 

In [29]:
# Confirming index structure
print(attempts_per_level.index.names)
print(completion_3w.index.names)
print(total_attempts_3w.index.names)
print(levels_completed_3w.index.names)
print(gap_features.index.names)
print(efficiency_3w.index.names)
print(total_inactivity_3w.index.names) 

['id', 'year']
['id', 'year']
['id', 'year']
['id', 'year']
['id', 'year']
['id', 'year']
['id', 'year']


In [30]:
# Checking for series features to convert to dataframe

print("attempts_per_level:", type(attempts_per_level))
print("completion_3w:", type(completion_3w))
print("total_attempts_3w:", type(total_attempts_3w))
print("levels_completed_3w:", type(levels_completed_3w))
print("efficiency_3w:", type(efficiency_3w))
print("gap_features:", type(gap_features))
print("total_inactivity_3w:", type(total_inactivity_3w))

attempts_per_level: <class 'pandas.core.frame.DataFrame'>
completion_3w: <class 'pandas.core.frame.DataFrame'>
total_attempts_3w: <class 'pandas.core.series.Series'>
levels_completed_3w: <class 'pandas.core.series.Series'>
efficiency_3w: <class 'pandas.core.series.Series'>
gap_features: <class 'pandas.core.frame.DataFrame'>
total_inactivity_3w: <class 'pandas.core.series.Series'>


In [31]:
total_attempts_3w = total_attempts_3w.to_frame()
levels_completed_3w = levels_completed_3w.to_frame()
efficiency_3w = efficiency_3w.to_frame()
total_inactivity_3w = total_inactivity_3w.to_frame()

In [32]:
student_features = master_students.copy()
student_features = student_features.merge(
    attempts_per_level,
    on=['id','year'],
    how='left'
)
student_features.head()

,id,year,num_atmp_l1,num_atmp_l2,num_atmp_l3,num_atmp_l4,num_atmp_l5,num_atmp_l6,num_atmp_l7,num_atmp_l8,num_atmp_l11
0,EWWZ8744,2014,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,FONP9675,2014,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,RNXB3999,2014,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,QSJZ8609,2014,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,VRMY8240,2014,1.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [33]:
student_features = student_features.merge(
    completion_3w,
    on=['id','year'],
    how='left'
)

In [34]:
student_features.head()

,id,year,num_atmp_l1,num_atmp_l2,num_atmp_l3,num_atmp_l4,num_atmp_l5,num_atmp_l6,num_atmp_l7,num_atmp_l8,num_atmp_l11,wk_comp_l1,wk_comp_l2,wk_comp_l3,wk_comp_l4,wk_comp_l5,wk_comp_l6,wk_comp_l7,wk_comp_l8
0,EWWZ8744,2014,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0
1,FONP9675,2014,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,0.0,0.0,0.0,0.0,0.0
2,RNXB3999,2014,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,0.0,0.0,0.0,0.0,0.0
3,QSJZ8609,2014,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,0.0,0.0,0.0,0.0,0.0
4,VRMY8240,2014,1.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,0.0,0.0,0.0,0.0,0.0


In [35]:
student_features = student_features.merge(
    total_attempts_3w,
    on=['id','year'],
    how='left'
)

In [36]:
student_features.shape, student_features.head()

((1154, 20),
          id  year  num_atmp_l1  num_atmp_l2  num_atmp_l3  num_atmp_l4  \
 0  EWWZ8744  2014          1.0          3.0          1.0          0.0   
 1  FONP9675  2014          1.0          1.0          1.0          0.0   
 2  RNXB3999  2014          1.0          1.0          1.0          0.0   
 3  QSJZ8609  2014          1.0          1.0          1.0          0.0   
 4  VRMY8240  2014          1.0          2.0          1.0          0.0   
 
    num_atmp_l5  num_atmp_l6  num_atmp_l7  num_atmp_l8  num_atmp_l11  \
 0          0.0          0.0          0.0          0.0           0.0   
 1          0.0          0.0          0.0          0.0           0.0   
 2          0.0          0.0          0.0          0.0           0.0   
 3          0.0          0.0          0.0          0.0           0.0   
 4          0.0          0.0          0.0          0.0           0.0   
 
    wk_comp_l1  wk_comp_l2  wk_comp_l3  wk_comp_l4  wk_comp_l5  wk_comp_l6  \
 0         1.0         3.0   

In [37]:
student_features = student_features.merge(
    levels_completed_3w,
    on=['id','year'],
    how='left'
)

In [38]:
student_features.shape, student_features.head()

((1154, 21),
          id  year  num_atmp_l1  num_atmp_l2  num_atmp_l3  num_atmp_l4  \
 0  EWWZ8744  2014          1.0          3.0          1.0          0.0   
 1  FONP9675  2014          1.0          1.0          1.0          0.0   
 2  RNXB3999  2014          1.0          1.0          1.0          0.0   
 3  QSJZ8609  2014          1.0          1.0          1.0          0.0   
 4  VRMY8240  2014          1.0          2.0          1.0          0.0   
 
    num_atmp_l5  num_atmp_l6  num_atmp_l7  num_atmp_l8  ...  wk_comp_l1  \
 0          0.0          0.0          0.0          0.0  ...         1.0   
 1          0.0          0.0          0.0          0.0  ...         1.0   
 2          0.0          0.0          0.0          0.0  ...         1.0   
 3          0.0          0.0          0.0          0.0  ...         1.0   
 4          0.0          0.0          0.0          0.0  ...         1.0   
 
    wk_comp_l2  wk_comp_l3  wk_comp_l4  wk_comp_l5  wk_comp_l6  wk_comp_l7  \
 0         

In [39]:
student_features = student_features.merge(
    efficiency_3w,
    on=['id','year'],
    how='left'
)

In [40]:
student_features.shape, student_features.head()

((1154, 22),
          id  year  num_atmp_l1  num_atmp_l2  num_atmp_l3  num_atmp_l4  \
 0  EWWZ8744  2014          1.0          3.0          1.0          0.0   
 1  FONP9675  2014          1.0          1.0          1.0          0.0   
 2  RNXB3999  2014          1.0          1.0          1.0          0.0   
 3  QSJZ8609  2014          1.0          1.0          1.0          0.0   
 4  VRMY8240  2014          1.0          2.0          1.0          0.0   
 
    num_atmp_l5  num_atmp_l6  num_atmp_l7  num_atmp_l8  ...  wk_comp_l2  \
 0          0.0          0.0          0.0          0.0  ...         3.0   
 1          0.0          0.0          0.0          0.0  ...         2.0   
 2          0.0          0.0          0.0          0.0  ...         2.0   
 3          0.0          0.0          0.0          0.0  ...         2.0   
 4          0.0          0.0          0.0          0.0  ...         2.0   
 
    wk_comp_l3  wk_comp_l4  wk_comp_l5  wk_comp_l6  wk_comp_l7  wk_comp_l8  \
 0         

In [41]:
student_features = student_features.merge(
    gap_features,
    on=['id','year'],
    how='left'
)

In [42]:
student_features.shape, student_features.head()

((1154, 29),
          id  year  num_atmp_l1  num_atmp_l2  num_atmp_l3  num_atmp_l4  \
 0  EWWZ8744  2014          1.0          3.0          1.0          0.0   
 1  FONP9675  2014          1.0          1.0          1.0          0.0   
 2  RNXB3999  2014          1.0          1.0          1.0          0.0   
 3  QSJZ8609  2014          1.0          1.0          1.0          0.0   
 4  VRMY8240  2014          1.0          2.0          1.0          0.0   
 
    num_atmp_l5  num_atmp_l6  num_atmp_l7  num_atmp_l8  ...  total_attempts_3w  \
 0          0.0          0.0          0.0          0.0  ...                5.0   
 1          0.0          0.0          0.0          0.0  ...                3.0   
 2          0.0          0.0          0.0          0.0  ...                3.0   
 3          0.0          0.0          0.0          0.0  ...                3.0   
 4          0.0          0.0          0.0          0.0  ...                4.0   
 
    levels_completed_3w  efficiency_3w  gap1to2

In [43]:
student_features = student_features.merge(
    total_inactivity_3w,
    on=['id','year'],
    how='left'
)

In [44]:
student_features.shape, student_features.head()

((1154, 30),
          id  year  num_atmp_l1  num_atmp_l2  num_atmp_l3  num_atmp_l4  \
 0  EWWZ8744  2014          1.0          3.0          1.0          0.0   
 1  FONP9675  2014          1.0          1.0          1.0          0.0   
 2  RNXB3999  2014          1.0          1.0          1.0          0.0   
 3  QSJZ8609  2014          1.0          1.0          1.0          0.0   
 4  VRMY8240  2014          1.0          2.0          1.0          0.0   
 
    num_atmp_l5  num_atmp_l6  num_atmp_l7  num_atmp_l8  ...  \
 0          0.0          0.0          0.0          0.0  ...   
 1          0.0          0.0          0.0          0.0  ...   
 2          0.0          0.0          0.0          0.0  ...   
 3          0.0          0.0          0.0          0.0  ...   
 4          0.0          0.0          0.0          0.0  ...   
 
    levels_completed_3w  efficiency_3w  gap1to2_3w  gap2to3_3w  gap3to4_3w  \
 0                  3.0           0.60         2.0         0.0         0.0   
 1   

In [45]:
# Adding zero for inactive students
student_features = student_features.fillna(0)

In [46]:
# Checks
print(student_features.shape)
print(student_features.columns)
print(student_features.isna().sum())

(1154, 30)
Index(['id', 'year', 'num_atmp_l1', 'num_atmp_l2', 'num_atmp_l3',
       'num_atmp_l4', 'num_atmp_l5', 'num_atmp_l6', 'num_atmp_l7',
       'num_atmp_l8', 'num_atmp_l11', 'wk_comp_l1', 'wk_comp_l2', 'wk_comp_l3',
       'wk_comp_l4', 'wk_comp_l5', 'wk_comp_l6', 'wk_comp_l7', 'wk_comp_l8',
       'total_attempts_3w', 'levels_completed_3w', 'efficiency_3w',
       'gap1to2_3w', 'gap2to3_3w', 'gap3to4_3w', 'gap4to5_3w', 'gap5to6_3w',
       'gap6to7_3w', 'gap7to8_3w', 'total_inactivity_3w'],
      dtype='object')
id                     0
year                   0
num_atmp_l1            0
num_atmp_l2            0
num_atmp_l3            0
num_atmp_l4            0
num_atmp_l5            0
num_atmp_l6            0
num_atmp_l7            0
num_atmp_l8            0
num_atmp_l11           0
wk_comp_l1             0
wk_comp_l2             0
wk_comp_l3             0
wk_comp_l4             0
wk_comp_l5             0
wk_comp_l6             0
wk_comp_l7             0
wk_comp_l8             

# The table shows no duplicate columns, naming inconsistencies, or missing values. Next, I will merge with course outcomes in the other tables in the spreadsheet.

# UPDATE

Included all levels recorded in Week of Completion for each level and number of attempts for each level in Week1-3

Removed all Nan values

Check for level recurrence after it was passed: 92 reattempts after mastery of a level were initially detected. Further inspection showed these cases were caused by students appearing in multiple cohort years. So a data integrity check was conducted to verify that students did not attempt mastery levels after successfully passing them within separate cohorts. After accounting for cohort year, only one instance of a post-mastery attempt was identified among 14,338 attempts. Inspection showed that the attempt number was reset, suggesting a system logging artefact rather than a genuine progression violation. Given its negligible frequency, the record was retained.

Is it better to focus on only levels that meet mastery learning criteria? 


# Checks Verifying That Only Data From 1st Three Weeks Is Used


In [47]:
# Checks

# Check if any completion week exceeds 3
(completion_3w > 3).sum()

wk_comp_l1    0
wk_comp_l2    0
wk_comp_l3    0
wk_comp_l4    0
wk_comp_l5    0
wk_comp_l6    0
wk_comp_l7    0
wk_comp_l8    0
dtype: int64

In [48]:
gap_features.describe()

,gap1to2_3w,gap2to3_3w,gap3to4_3w,gap4to5_3w,gap5to6_3w,gap6to7_3w,gap7to8_3w
count,1152.000000,1152.000000,1152.000000,1152.000000,1152.0,1152.0,1152.0
mean,0.872396,0.325521,0.036458,0.007812,0.0,0.0,0.0
std,0.702634,0.476128,0.187509,0.088081,0.0,0.0,0.0
min,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
25%,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
50%,1.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
75%,1.000000,1.000000,0.000000,0.000000,0.0,0.0,0.0
max,2.000000,2.000000,1.000000,1.000000,0.0,0.0,0.0


In [49]:
early_attempts['week'].max()

np.int64(3)

In [50]:
student_features[student_features['total_attempts_3w'] == 0].shape

(21, 30)

## Export Feature Table


In [51]:
# Export early-window student features for the next notebook
student_features.to_excel("student_features_early3w.xlsx", index=False)
print("Saved: student_features_early3w.xlsx")


Saved: student_features_early3w.xlsx
